# 通用DQN + double DQN + dueling DQN

https://walkinglabs.github.io/hands-on-modern-rl/chapter07_dqn/dqn-components

离散空间当中使用的算法

# 通用DQN

DQN 的一个核心思想是定义两个能够根据当前环境输出每个动作 Q 值的网络，称为 Q 网络。同时会设有两个 Q 网络：

1. 第一个 Q 网络作为 Target 网络（目标网络）
2. 第二个网络是训练网络 Q 网络

训练过程中，Q 训练网络会将当前的环境输入进去，以及当前选择的一个动作，来算出当前的一个最大 Q 值。另外一组是当前的奖励加上 Target 网络输入 next state 后所获得的一个最大 Q 值。

两者之间让当前训练网络不断去追赶这个目标奖励函数。每过一定轮数以后，让 Target 网络来复制当前训练网络的权重，让训练网络重新进行追赶，就相当于是不断用当前的奖励，使得 Target 和 Q 网络不断变得更加精准。

# DQN (Deep Q-Network) 核心原理笔记

## 1. 算法概述

DQN 是将深度学习（Deep Learning）与 Q-Learning 结合的经典强化学习算法。它利用深度神经网络强大的拟合能力，解决了传统 Q-Learning 在状态空间过大时无法构建 Q 表的问题。

---

## 2. 核心组件与符号定义

- $s$: 当前环境状态 (State)
- $a$: 智能体执行的动作 (Action)
- $r$: 环境反馈的即时奖励 (Reward)
- $s'$: 执行动作后转移到的下一个状态 (Next State)
- $\gamma$: 折扣因子 ($0 \le \gamma \le 1$)，表示对未来奖励的重视程度
- $Q(s, a; \theta)$: **评估网络 (Online Network)**，参数为 $\theta$，用于估计当前状态动作对的 Q 值。
- $Q(s', a'; \theta^-)$: **目标网络 (Target Network)**，参数为 $\theta^-$，用于计算目标 Q 值，参数在一段时间内保持冻结。

---

## 3. 核心机制

### 3.1 目标网络机制 (Target Network)

为了打破传统 Q-Learning 中“目标值随网络参数实时更新”导致的训练不稳定性，DQN 引入了目标网络。目标网络的参数 $\theta^-$ 每隔 $C$ 步才从评估网络同步一次：

$\theta^- \leftarrow \theta$

### 3.2 经验回放机制 (Experience Replay)

将智能体与环境交互产生的数据 $(s, a, r, s')$ 存入经验回放池 $\mathcal{D}$。训练时，从 $\mathcal{D}$ 中**随机采样**一个 Mini-batch 进行梯度下降。这打破了时间序列数据的强相关性，使数据近似满足独立同分布 (I.I.D.) 假设。

---

## 4. 核心公式推导

### 4.1 贝尔曼方程 (Bellman Equation)

Q 值的理论更新依据是贝尔曼最优方程，即当前 Q 值等于即时奖励加上未来最大折扣 Q 值：

$Q^*(s, a) = r + \gamma \max_{a'} Q^*(s', a')$

### 4.2 目标值 (Target) 计算

在 DQN 中，利用**目标网络**来计算稳定的目标值 $y$：

$y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$

### 4.3 损失函数 (Loss Function)

评估网络通过最小化其预测值与目标值之间的均方误差 (MSE) 来进行参数 $\theta$ 的更新：

$L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ \left( y - Q(s, a; \theta) \right)^2 \right]$

展开即为：

$L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ \left( r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta) \right)^2 \right]$

**核心公式备注：**

### 4.4 梯度更新

对损失函数求关于 $\theta$ 的梯度，并使用随机梯度下降 (SGD) 更新评估网络参数：

$\theta \leftarrow \theta - \alpha \nabla_\theta L(\theta)$

*(注：*$\alpha$* 为学习率)*

---

## 5. 算法执行流程总结

1. **交互与存储**：评估网络根据 $\epsilon$-greedy 策略选择动作 $a$，与环境交互获得 $(s, a, r, s')$，存入经验池 $\mathcal{D}$。
2. **采样**：从 $\mathcal{D}$ 中随机采样一个 Mini-batch。
3. **计算目标**：利用**目标网络**计算每个样本的目标值 $y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$。
4. **更新评估网络**：计算 Loss，通过反向传播更新评估网络参数 $\theta$。
5. **同步目标网络**：每隔 $C$ 步，将评估网络的参数 $\theta$ 复制给目标网络 $\theta^-$。


# 代码

## Q网络
它主要用来让智能体（Agent）在复杂的环境中学会如何做出最优的，是说根据当前环境来输出每一个动作的Q值

In [1]:
import torch
import torch.nn as nn

class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),   # 输入层 → 隐藏层
            nn.ReLU(),                          # 激活函数
            nn.Linear(hidden_dim, hidden_dim),  # 隐藏层 → 隐藏层
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),  # 隐藏层 → 输出层
        )

    def forward(self, x):
        return self.net(x)  # 输出形状: (batch_size, action_dim)

## 经验回放

准备一个容器存历史经验，训练时随机采样打破相关性。智能体每走一步就产生一条转移 (s,a,r,s 
′
 ,d)，回放池就是把这些转移攒起来。

In [2]:
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(states),      # (B, state_dim)
            torch.LongTensor(actions),       # (B,)
            torch.FloatTensor(rewards),      # (B,)
            torch.FloatTensor(next_states),  # (B, state_dim)
            torch.FloatTensor(dones),        # (B,)
        )

    def __len__(self):
        return len(self.buffer)

## 定义DQNAgent

In [4]:
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_

class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99): # Adam 的默认学习率，也是 DQN 常用起点
        self.action_dim = action_dim
        self.gamma = gamma # 折扣因子。0.99 让 100 步后的奖励衰减到 0.99^100 ≈0.366，兼顾眼前和未来。0.9 太短视，0.999 方差太大。

        # Q 网络（学生）和目标网络（阅卷老师）
        self.q_net = QNetwork(state_dim, action_dim)
        self.target_net = QNetwork(state_dim, action_dim)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(capacity=10000)

1. 传统DQN有一个 Q 网络和一个差别网络，它们的一个目标都是用来选择一个合适的动作，然后拿出这个动作的一个 Q 值估计。
2. double DQN 解决的是传统DQN又当裁判又当运动员的问题：它将 Q 网络和 Target 网络同样存在，但是 Q 网络我们都是在用来评估一个合适的动作，但是 Q 网络更多的是偏向于选择 Target 的网络，它的代码更多偏向于评估它的权重。
3. Dueling DQN：传统 DQN 会评估优势 Q（S,a），其中包含环境因素 S 和动作因素 A。其思路是使用两个网络分别评估这两个参数。$$Q(s,a) = V(s) + A(s,a)$$ 

In [ ]:
# 传统DQN
def update(self, batch_size):
    """核心更新：一个 batch 的前向传播 + 反向传播"""
    if len(self.buffer) < batch_size:
        return 0.0

    # 从回放池采样一个 batch
    states, actions, rewards, next_states, dones = self.buffer.sample(batch_size)

    # Q 网络前向传播
    q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # 目标网络前向传播
    with torch.no_grad():
        next_q_max = self.target_net(next_states).max(dim=1)[0]
        targets = rewards + self.gamma * next_q_max * (1 - dones)

    # 计算 MSE Loss
    loss = nn.MSELoss()(q_values, targets)

    # 反向传播与参数更新
    self.optimizer.zero_grad()
    loss.backward()
    clip_grad_norm_(self.q_net.parameters(), max_norm=10)
    self.optimizer.step()

    return loss.item()


In [6]:
def select_action(self, state, epsilon):
    """ε-greedy 动作选择"""
    if random.random() < epsilon:
        return random.randint(0, self.action_dim - 1)
    with torch.no_grad():
        q_values = self.q_net(torch.FloatTensor(state).unsqueeze(0))
    return q_values.argmax(dim=1).item()

def update_target(self):
    """硬更新：将 Q 网络参数复制到目标网络"""
    self.target_net.load_state_dict(self.q_net.state_dict())

In [ ]:
import gymnasium as gym

num_episodes = 500
batch_size = 64
epsilon_start, epsilon_end, epsilon_decay = 1.0, 0.01, 0.995
target_update_freq = 10

env = gym.make("CartPole-v1")
agent = DQNAgent(state_dim=4, action_dim=2)
epsilon = epsilon_start

for episode in range(num_episodes):
    state, _ = env.reset()
    while True:
        action = agent.select_action(state, epsilon)
        next_state, reward, done, truncated, _ = env.step(action)
        agent.buffer.push(state, action, reward, next_state, float(done))
        agent.update(batch_size)
        state = next_state
        if done or truncated:
            break

    epsilon = max(epsilon_end, epsilon * epsilon_decay)
    if (episode + 1) % target_update_freq == 0:
        agent.update_target()

# Double DQN

double DQN 解决的是传统DQN又当裁判又当运动员的问题：它将 Q 网络和 Target 网络同样存在，但是 Q 网络我们都是在用来评估一个合适的动作，但是 Q 网络更多的是偏向于选择 Target 的网络，它的代码更多偏向于评估它的权重。

这里吧update网络修改下就行

In [ ]:

# double DQN 是一种改写， Q网络用于选择动作，Target Network用于评估权重。
def update_double_DQN(self, batch_size):
    """核心更新：一个 batch 的前向传播 + 反向传播 (Double DQN)"""
    if len(self.buffer) < batch_size:
        return 0.0

    # 从回放池采样一个 batch
    states, actions, rewards, next_states, dones = self.buffer.sample(batch_size)

    # Q 网络前向传播：获取当前状态下执行动作的 Q 值
    q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # ================= 修改的核心部分 =================
    with torch.no_grad():
        # 1. 运动员 (q_net) 负责“选择”动作：选出 next_states 下 Q 值最大的那个动作
        best_next_actions = self.q_net(next_states).argmax(dim=1, keepdim=True)
        
        # 2. 裁判员 (target_net) 负责“评估”动作：获取这个被选中动作的目标 Q 值
        next_q_selected = self.target_net(next_states).gather(1, best_next_actions).squeeze(1)
        
        # 3. 计算 TD Target
        targets = rewards + self.gamma * next_q_selected * (1 - dones)
    # ===================================================

    # 计算 MSE Loss
    loss = nn.MSELoss()(q_values, targets)

    # 反向传播与参数更新
    self.optimizer.zero_grad()
    loss.backward()
    clip_grad_norm_(self.q_net.parameters(), max_norm=10)
    self.optimizer.step()

    return loss.item()

# dueling DQN

In [ ]:
# 传统DQN它就是一个网络直接评估出来它的一个Q值

import torch
import torch.nn as nn

class StandardDQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            # 最后一层直接输出所有动作的预测分数
            nn.Linear(128, action_dim) 
        )

    def forward(self, x):
        # 一步到位输出 Q(s,a)
        return self.network(x)

In [ ]:
class DuelingDQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        
        # 第一部分：共享特征层（提取当前环境的基本特征）
        self.feature_layer = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU()
        )
        
        # 第二部分：价值分支 V(s) —— 评估状态本身的“底薪”
        # 注意：这里的最后一层输出维度是 1，因为它只打一个总分
        self.value_stream = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  
        )
        
        # 第三部分：优势分支 A(s,a) —— 评估每个动作相对的“绩效”
        # 注意：这里的输出维度是 action_dim，有几个动作就有几个优势值
        self.advantage_stream = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim) 
        )

    def forward(self, x):
        # 1. 先过一遍共享特征层
        features = self.feature_layer(x)
        
        # 2. 分别算出 V 和 A
        V = self.value_stream(features)   # 形状: [batch_size, 1]
        A = self.advantage_stream(features) # 形状: [batch_size, action_dim]
        
        # 3. 核心合成公式：Q(s,a) = V(s) + A(s,a) - mean(A)
        # 为什么要 keepdim=True？为了让减去平均值的时候，矩阵维度对齐（广播机制）
        Q = V + A - A.mean(dim=1, keepdim=True)
        
        return Q